In [1]:
import sys
import os
import re
import numpy as np
import pandas as pd
import Levenshtein
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Load Pre-Cleaned Training Data
train_s1 = pd.read_csv("D:/MLChallenge/ml-challenge-26/notebooks/jeff/output/cleaned_data/train_source1_cleaned.tsv", sep="\t")
train_s2 = pd.read_csv("D:/MLChallenge/ml-challenge-26/notebooks/jeff/output/cleaned_data/train_source2_cleaned.tsv", sep="\t")
train_s3 = pd.read_csv("D:/MLChallenge/ml-challenge-26/notebooks/jeff/output/cleaned_data/train_source3_cleaned.tsv", sep="\t")
train_gt = pd.read_csv("D:/MLChallenge/ml-challenge-26/notebooks/jeff/dataset/train/train_ground_truth.tsv", sep="\t")

# Load Pre-Cleaned Test Data
test_s1 = pd.read_csv("D:/MLChallenge/ml-challenge-26/notebooks/jeff/output/cleaned_data/test_source1_cleaned.tsv", sep="\t")
test_s2 = pd.read_csv("D:/MLChallenge/ml-challenge-26/notebooks/jeff/output/cleaned_data/test_source2_cleaned.tsv", sep="\t")
test_s3 = pd.read_csv("D:/MLChallenge/ml-challenge-26/notebooks/jeff/output/cleaned_data/test_source3_cleaned.tsv", sep="\t")

# Ensure all text columns are treated as standard strings for similarity functions
for df in [train_s1, train_s2, train_s3, test_s1, test_s2, test_s3]:
    df['clean_name'] = df['clean_name'].astype(str)
    df['clean_address'] = df['clean_address'].astype(str)

print("Datasets loaded successfully and ready for candidate generation!")

Datasets loaded successfully and ready for candidate generation!


In [ ]:
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from pathlib import Path
import time
import gc

STOP_LEGAL_TOKENS = {'inc', 'llc', 'co', 'corp', 'pvt', 'ltd', 'the', 'and'}

def sorted_name_key(clean_name, n_chars=4):
    """Order-invariant sub-block key: alphabetically-first non-trivial token,
    truncated. Handles word-order transpositions and stripped legal suffixes."""
    tokens = [t for t in str(clean_name).split() if t not in STOP_LEGAL_TOKENS]
    if not tokens:
        tokens = str(clean_name).split()
    if not tokens:
        return "__EMPTY__"
    return sorted(tokens)[0][:n_chars]

def addr_digit_key(clean_address, min_len=4):
    runs = re.findall(r'\d{' + str(min_len) + r',}', str(clean_address))
    return max(runs, key=len) if runs else None 

def tfidf_topk_within_block(s1_sub, pool_df, threshold, top_k, vec_kwargs):
    pool_names = pool_df['clean_name'].fillna('').astype(str)
    s1_names = s1_sub['clean_name'].fillna('').astype(str)
    if not pool_names.str.strip().any() or not s1_names.str.strip().any():
        return []

    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 4), **vec_kwargs)
    tfidf_pool = vectorizer.fit_transform(pool_names)
    tfidf_s1 = vectorizer.transform(s1_names)

    k = min(top_k, tfidf_pool.shape[0])
    nn = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='brute')
    nn.fit(tfidf_pool)
    distances, indices = nn.kneighbors(tfidf_s1)
    similarities = 1 - distances

    s1_ids = s1_sub['entity_id'].to_numpy()
    pool_ids = pool_df['entity_id'].to_numpy()

    out = []
    for i in range(len(s1_ids)):
        keep = similarities[i] >= threshold
        for j, sim in zip(indices[i][keep], similarities[i][keep]):
            out.append((s1_ids[i], pool_ids[j], float(sim)))
    return out

def generate_name_key_candidates(s1_sub, pool_df, threshold, top_k, name_key_chars, vectorizer_max_features):
    s1_sub = s1_sub.copy()
    pool_df = pool_df.copy()
    s1_sub['_name_key'] = s1_sub['clean_name'].map(lambda n: sorted_name_key(n, name_key_chars))
    pool_df['_name_key'] = pool_df['clean_name'].map(lambda n: sorted_name_key(n, name_key_chars))

    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 4), max_features=vectorizer_max_features)
    pool_matrix_all = vectorizer.fit_transform(pool_df['clean_name'].fillna('').astype(str))
    s1_matrix_all = vectorizer.transform(s1_sub['clean_name'].fillna('').astype(str))

    pool_groups = pool_df.groupby('_name_key').indices
    s1_groups = s1_sub.groupby('_name_key').indices

    s1_ids_all = s1_sub['entity_id'].to_numpy()
    pool_ids_all = pool_df['entity_id'].to_numpy()

    pairs = []
    for key_val, s1_idx in s1_groups.items():
        pool_idx = pool_groups.get(key_val)
        if pool_idx is None or len(pool_idx) == 0:
            continue
        s1_block = s1_matrix_all[s1_idx]
        pool_block = pool_matrix_all[pool_idx]
        k = min(top_k, pool_block.shape[0])
        nn = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='brute')
        nn.fit(pool_block)
        distances, indices = nn.kneighbors(s1_block)
        similarities = 1 - distances
        block_s1_ids = s1_ids_all[s1_idx]
        block_pool_ids = pool_ids_all[pool_idx]
        for i in range(len(block_s1_ids)):
            keep = similarities[i] >= threshold
            for j, sim in zip(indices[i][keep], similarities[i][keep]):
                pairs.append((block_s1_ids[i], block_pool_ids[j], float(sim)))
    return pairs

def generate_candidates_v2(
    s1_df, s2_df, s3_df,
    threshold=0.20,
    top_k=15,
    max_pool_size=40000,
    name_key_chars=4,
    vectorizer_max_features=50000,
    verbose=True,
):
    s1_df, s2_df, s3_df = s1_df.copy(), s2_df.copy(), s3_df.copy()
    fill_val = "__UNKNOWN__"
    for df in (s1_df, s2_df, s3_df):
        df['_country_key'] = df['country'].fillna(fill_val)

    pair_sim = {}     
    pair_source = defaultdict(set) 

    countries = s1_df['_country_key'].unique()
    if verbose:
        print(f"Processing {len(countries)} country blocks...")

    for country in countries:
        s1_sub = s1_df[s1_df['_country_key'] == country]
        if s1_sub.empty:
            continue

        if country == fill_val:
            pool_df = pd.concat([s2_df, s3_df], ignore_index=True)
        else:
            s2_sub = s2_df[(s2_df['_country_key'] == country) | (s2_df['_country_key'] == fill_val)]
            s3_sub = s3_df[(s3_df['_country_key'] == country) | (s3_df['_country_key'] == fill_val)]
            pool_df = pd.concat([s2_sub, s3_sub], ignore_index=True)

        if pool_df.empty:
            continue

        vec_kwargs = {'max_features': vectorizer_max_features}

        # ---- Stage A/B: name similarity ----
        if len(pool_df) <= max_pool_size:
            if verbose:
                print(f"[{country}] pool={len(pool_df)} -> single TF-IDF/NN pass")
            pairs = tfidf_topk_within_block(s1_sub, pool_df, threshold, top_k, vec_kwargs)
        else:
            if verbose:
                print(f"[{country}] pool={len(pool_df)} > {max_pool_size} -> sub-blocking by name key")
            pairs = generate_name_key_candidates(
                s1_sub, pool_df, threshold, top_k, name_key_chars, vectorizer_max_features
            )

        for s1_id, cand_id, sim in pairs:
            key = (s1_id, cand_id)
            if key not in pair_sim or sim > pair_sim[key]:
                pair_sim[key] = sim
            pair_source[key].add('name')

        # ---- Stage C: address-digit exact match ----
        MAX_ADDR_GROUP_PRODUCT = 2_000_000   

        s1_addr = s1_sub[['entity_id', 'clean_address']].copy()
        s1_addr['_addr_key'] = s1_addr['clean_address'].map(addr_digit_key)
        s1_addr = s1_addr.dropna(subset=['_addr_key'])

        pool_addr = pool_df[['entity_id', 'clean_address']].copy()
        pool_addr['_addr_key'] = pool_addr['clean_address'].map(addr_digit_key)
        pool_addr = pool_addr.dropna(subset=['_addr_key'])

        if not s1_addr.empty and not pool_addr.empty:
            s1_counts = s1_addr['_addr_key'].value_counts()
            pool_counts = pool_addr['_addr_key'].value_counts()
            common_keys = s1_counts.index.intersection(pool_counts.index)
            fan_out = s1_counts.loc[common_keys] * pool_counts.loc[common_keys]
            safe_keys = fan_out[fan_out <= MAX_ADDR_GROUP_PRODUCT].index

            s1_addr_safe = s1_addr[s1_addr['_addr_key'].isin(safe_keys)]
            pool_addr_safe = pool_addr[pool_addr['_addr_key'].isin(safe_keys)]

            merged = s1_addr_safe.merge(pool_addr_safe, on='_addr_key', suffixes=('_s1', '_pool'))
            for s1_id, cand_id in zip(merged['entity_id_s1'], merged['entity_id_pool']):
                key = (s1_id, cand_id)
                if key not in pair_sim:
                    pair_sim[key] = np.nan
                pair_source[key].add('address')

    records = [
        {
            'source1_entity_id': s1_id,
            'candidate_entity_id': cand_id,
            'sim_score': sim,
            'source': '+'.join(sorted(pair_source[(s1_id, cand_id)])),
        }
        for (s1_id, cand_id), sim in pair_sim.items()
    ]
    return pd.DataFrame(records)

def check_blocking_recall(candidates_df, gt_df):
    print("Checking blocking recall...")
    gt_dict = {}
    for _, row in gt_df.iterrows():
        matches = str(row['matched_entity_ids']).split(',') if pd.notna(row['matched_entity_ids']) else []
        gt_dict[str(row['source1_entity_id'])] = set(str(m).strip() for m in matches if str(m).strip())

    candidates_df = candidates_df.copy()
    candidates_df['source1_entity_id'] = candidates_df['source1_entity_id'].astype(str)
    candidates_df['candidate_entity_id'] = candidates_df['candidate_entity_id'].astype(str)

    cand_dict = candidates_df.groupby('source1_entity_id')['candidate_entity_id'].apply(set).to_dict()

    total_gt_matches = 0
    recovered = 0
    fully_missed_entities = 0

    for s1_id, true_matches in gt_dict.items():
        if not true_matches:
            continue
        candidate_set = cand_dict.get(s1_id, set())
        hit = true_matches & candidate_set
        total_gt_matches += len(true_matches)
        recovered += len(hit)
        if not hit:
            fully_missed_entities += 1

    recall = recovered / total_gt_matches if total_gt_matches else 0.0
    print(f"Blocking recall: {recovered}/{total_gt_matches} = {recall:.4f}")
    print(f"Entities with ZERO surviving true match: {fully_missed_entities}")
    return recall

required_dataframes = ["train_s1", "train_s2", "train_s3", "train_gt"]
missing_data = any(name not in globals() for name in required_dataframes)
missing_clean_names = not missing_data and any(
    "clean_name" not in globals()[name].columns
    for name in ["train_s1", "train_s2", "train_s3"]
)

if missing_data or missing_clean_names:
    print("Training data is not initialized. Loading the prepared training files...")
    cleaned_dir_candidates = [
        Path("notebooks/joe/output/cleaned_data"),
        Path("output/cleaned_data"),
    ]
    cleaned_dir = next((path for path in cleaned_dir_candidates if path.exists()), None)
    if cleaned_dir is None:
        raise FileNotFoundError("Prepared files were not found.")

    dataset_dir = Path(r"C:\Users\chinm\Downloads\dataset\student_resource\dataset")
    train_dir = dataset_dir / "train"

    train_s1 = pd.read_csv(cleaned_dir / "train_source1_cleaned.tsv", sep="\t")
    train_s2 = pd.read_csv(cleaned_dir / "train_source2_cleaned.tsv", sep="\t")
    train_s3 = pd.read_csv(cleaned_dir / "train_source3_cleaned.tsv", sep="\t")
    train_gt = pd.read_csv(train_dir / "train_ground_truth.tsv", sep="\t")

print("Generating training candidates in chunks for full train_s1...")

# ---- CHUNKED PROCESSING FOR THE FULL 2.2M TRAIN_S1 ----
CHUNK_SIZE = 50000
train_raw_path = Path("output/train_candidates_raw.tsv")
train_done_dir = Path("output/train_chunks_done")
train_done_dir.mkdir(parents=True, exist_ok=True)

if train_raw_path.exists():
    train_raw_path.unlink()

n_chunks = (len(train_s1) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Processing full train_s1 ({len(train_s1)} rows) in {n_chunks} chunks of {CHUNK_SIZE}")

t0 = time.time()
for i in range(n_chunks):
    marker = train_done_dir / f"{i}.done"
    if marker.exists():
        continue

    start, end = i * CHUNK_SIZE, min((i + 1) * CHUNK_SIZE, len(train_s1))
    chunk_s1 = train_s1.iloc[start:end]

    chunk_candidates = generate_candidates_v2(
        chunk_s1, train_s2, train_s3,
        threshold=0.20, top_k=15, max_pool_size=40000,
        verbose=False,
    )

    chunk_candidates.to_csv(
        train_raw_path, sep="\t", mode="a",
        header=not train_raw_path.exists() or i == 0, index=False
    )
    marker.touch()

    print(f"Train Chunk {i+1}/{n_chunks} rows[{start}:{end}] -> {len(chunk_candidates)} pairs")

print(f"Full training candidate generation complete in {time.time() - t0:.1f}s")

### Blocking Helpers and Candidate Generation Functions

In [ ]:
N_SHARDS = 20
train_shard_paths = [Path(f"output/train_shard_{i}.tsv") for i in range(N_SHARDS)]
for p in train_shard_paths:
    if p.exists():
        p.unlink()

def shard_of(s1_id, n=N_SHARDS):
    return int(hashlib.md5(str(s1_id).encode()).hexdigest(), 16) % n

t0 = time.time()
reader = pd.read_csv("output/train_candidates_raw.tsv", sep="\t", chunksize=2_000_000)
for i, chunk in enumerate(reader):
    chunk['_shard'] = chunk['source1_entity_id'].map(shard_of)
    for s in range(N_SHARDS):
        sub = chunk[chunk['_shard'] == s].drop(columns='_shard')
        if not sub.empty:
            header = not train_shard_paths[s].exists()
            sub.to_csv(train_shard_paths[s], sep="\t", mode="a", header=header, index=False)
    print(f"Partitioned train read-block {i} ({time.time()-t0:.1f}s)")
    del chunk
    gc.collect()

print("Training candidate sharding complete.")

In [ ]:
def jaccard_tokens(a, b):
    set_a, set_b = set(a.split()), set(b.split())
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def ngram_set(s, n=3):
    return set(s[i:i+n] for i in range(len(s) - n + 1)) if len(s) >= n else {s}

def ngram_jaccard(a, b, n=3):
    set_a, set_b = ngram_set(a, n), ngram_set(b, n)
    if not set_a and not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def build_lookup(entity_ids, *source_dfs):
    needed = set(entity_ids)
    parts = []
    for df in source_dfs:
        sub = df[df['entity_id'].isin(needed)][['entity_id', 'clean_name', 'clean_address', 'country']]
        parts.append(sub)
    return pd.concat(parts, ignore_index=True).drop_duplicates('entity_id').set_index('entity_id')

def build_feature_table(candidates_df, s1_df, s2_df, s3_df, gt_df=None):
    df = candidates_df.copy()

    s1_lookup = build_lookup(df['source1_entity_id'].unique(), s1_df)
    pool_lookup = build_lookup(df['candidate_entity_id'].unique(), s2_df, s3_df)

    df = df.merge(s1_lookup.add_suffix('_s1'), left_on='source1_entity_id', right_index=True, how='left')
    df = df.merge(pool_lookup.add_suffix('_cand'), left_on='candidate_entity_id', right_index=True, how='left')

    for col in ['clean_name_s1', 'clean_name_cand', 'clean_address_s1', 'clean_address_cand']:
        df[col] = df[col].fillna('')

    df['name_lev_ratio'] = [Levenshtein.ratio(a, b) for a, b in zip(df['clean_name_s1'], df['clean_name_cand'])]
    df['name_jaccard'] = [ngram_jaccard(a, b) for a, b in zip(df['clean_name_s1'], df['clean_name_cand'])]
    df['name_len_diff'] = (df['clean_name_s1'].str.len() - df['clean_name_cand'].str.len()).abs()

    df['addr_lev_ratio'] = [Levenshtein.ratio(a, b) for a, b in zip(df['clean_address_s1'], df['clean_address_cand'])]
    df['addr_jaccard'] = [ngram_jaccard(a, b) for a, b in zip(df['clean_address_s1'], df['clean_address_cand'])]

    df['country_match'] = (df['country_s1'] == df['country_cand']).astype(int)

    df['tfidf_sim_score'] = df['sim_score']              
    df['found_by_name'] = df['source'].str.contains('name').astype(int)
    df['found_by_address'] = df['source'].str.contains('address').astype(int)

    if gt_df is not None:
        gt_pairs = set()
        for _, row in gt_df.iterrows():
            if pd.notna(row['matched_entity_ids']) and str(row['matched_entity_ids']).strip():
                for mid in str(row['matched_entity_ids']).split(','):
                    gt_pairs.add((row['source1_entity_id'], mid.strip()))
        df['label'] = [1 if (s1, c) in gt_pairs else 0
                        for s1, c in zip(df['source1_entity_id'], df['candidate_entity_id'])]

    return df

# Split train_s1 entity IDs into 80% train, 20% validation
all_train_ids = train_s1['entity_id'].unique()
train_ids, val_ids = train_test_split(all_train_ids, test_size=0.2, random_state=42)
train_ids_set, val_ids_set = set(train_ids), set(val_ids)

train_feat_list = []
val_feat_list = []

FEATURE_COLS = [
    'name_lev_ratio', 'name_jaccard', 'name_len_diff',
    'addr_lev_ratio', 'addr_jaccard',
    'country_match', 'tfidf_sim_score', 'found_by_name', 'found_by_address',
]

for s in range(N_SHARDS):
    shard_df = pd.read_csv(train_shard_paths[s], sep="\t")
    if shard_df.empty:
        continue
    
    feat_shard = build_feature_table(shard_df, train_s1, train_s2, train_s3, gt_df=train_gt)
    
    train_feat_list.append(feat_shard[feat_shard['source1_entity_id'].isin(train_ids_set)])
    val_feat_list.append(feat_shard[feat_shard['source1_entity_id'].isin(val_ids_set)])
    
    print(f"Processed training shard {s}/{N_SHARDS-1}")
    del shard_df, feat_shard
    gc.collect()

train_feat = pd.concat(train_feat_list, ignore_index=True)
val_feat = pd.concat(val_feat_list, ignore_index=True)

X_train, y_train = train_feat[FEATURE_COLS], train_feat['label']
X_val, y_val = val_feat[FEATURE_COLS], val_feat['label']
print(f"X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {pos_weight:.1f}")

model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    random_state=42,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=20), lgb.log_evaluation(period=10)],
)
print(f"Real best iteration: {model.best_iteration_}")
print(f"Real best val AUC: {model.best_score_['valid_0']['auc']:.4f}")

# Save the fully trained model
joblib.dump(model, 'output/lgbm_baseline_model.pkl')
print("Model saved to output/lgbm_baseline_model.pkl")